In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path

# ── Config ────────────────────────────────────────────────────────────────────
# Adjust INPUT_DIR to the actual slug shown in Kaggle UI after adding
# Notebook 19's output via "+ Add Input → Notebook Output Files"
INPUT_DIR   = Path("/kaggle/input/orbit-wars-pipeline-output")
OUTPUT_DIR  = Path("/kaggle/working")

BATCH_SIZE  = 1024
EPOCHS      = 150
LR          = 1e-3
PATIENCE_LR = 10    # ReduceLROnPlateau: halve LR after this many non-improving epochs
PATIENCE_ES = 20    # Early stopping: halt after this many non-improving epochs
LR_FACTOR   = 0.5
RANDOM_SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
class OrbitMLP(nn.Module):
    """
    Multi-task MLP for imitation learning.
    Heads: from (45 classes, 44=stop), to (44 classes), ships (regression).
    """
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.BatchNorm1d(input_dim),
            nn.Linear(input_dim, 1024), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(1024, 512),       nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(512, 256),        nn.ReLU(),
        )
        self.head_from  = nn.Linear(256, 45)
        self.head_to    = nn.Linear(256, 44)
        self.head_ships = nn.Linear(256, 1)

    def forward(self, x):
        h = self.encoder(x)
        return self.head_from(h), self.head_to(h), self.head_ships(h).squeeze(-1)

In [ ]:
X       = np.load(INPUT_DIR / "X.npy")
y_from  = np.load(INPUT_DIR / "y_from.npy")
y_to    = np.load(INPUT_DIR / "y_to.npy")
y_ships = np.load(INPUT_DIR / "y_ships.npy")

# Compute mask BEFORE nan_to_num: stop rows have y_to=NaN, action rows have y_to=slot (0-43)
# Also filter out any -1 invalid slots that may have slipped through (safety net)
valid   = (y_from >= 0) & (np.isnan(y_to) | (y_to >= 0))
X       = X[valid]
y_from  = y_from[valid]
y_to    = y_to[valid]
y_ships = y_ships[valid]
if valid.sum() < len(valid):
    print(f"Filtered {(~valid).sum()} rows with invalid slot (-1)")

mask_ns = (y_from != 44) & (~np.isnan(y_to))   # non-stop rows with valid destination

y_to    = np.nan_to_num(y_to,    nan=0.0).astype(np.int64)
y_ships = np.nan_to_num(y_ships, nan=0.0)

input_dim = X.shape[1]
print(f"Loaded  X={X.shape}  y_from={y_from.shape}")
print(f"Non-stop rows: {mask_ns.sum():,} / {len(X):,}")

X_t      = torch.tensor(X,                        dtype=torch.float32)
yf_t     = torch.tensor(y_from.astype(np.int64),  dtype=torch.long)
yt_t     = torch.tensor(y_to,                     dtype=torch.long)
ys_t     = torch.tensor(y_ships.astype(np.float32), dtype=torch.float32)
mk_t     = torch.tensor(mask_ns,                  dtype=torch.bool)

g   = torch.Generator().manual_seed(RANDOM_SEED)
idx = torch.randperm(len(X_t), generator=g)
n_val = int(0.2 * len(X_t))
tr_idx, va_idx = idx[n_val:], idx[:n_val]

tr_ds = TensorDataset(X_t[tr_idx], yf_t[tr_idx], yt_t[tr_idx], ys_t[tr_idx], mk_t[tr_idx])
va_ds = TensorDataset(X_t[va_idx], yf_t[va_idx], yt_t[va_idx], ys_t[va_idx], mk_t[va_idx])

tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True)
va_loader = DataLoader(va_ds, batch_size=BATCH_SIZE)
print(f"Train batches: {len(tr_loader)}  Val batches: {len(va_loader)}")

In [ ]:
model     = OrbitMLP(input_dim).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=PATIENCE_LR, factor=LR_FACTOR
)
ce_from = nn.CrossEntropyLoss()
ce_to   = nn.CrossEntropyLoss()


def eval_epoch(loader):
    model.eval()
    acc_f = acc_t = mae_s = 0.0
    n = n_ns = 0
    with torch.no_grad():
        for xb, yf, yt, ys, mk in loader:
            xb, yf, yt, ys, mk = xb.to(DEVICE), yf.to(DEVICE), yt.to(DEVICE), ys.to(DEVICE), mk.to(DEVICE)
            lf, lt, ps = model(xb)
            acc_f += (lf.argmax(1) == yf).float().mean().item()
            n += 1
            if mk.any():
                acc_t += (lt[mk].argmax(1) == yt[mk]).float().mean().item()
                mae_s += (ps[mk] - ys[mk]).abs().mean().item()
                n_ns  += 1
    return acc_f / max(n, 1), acc_t / max(n_ns, 1), mae_s / max(n_ns, 1)


best_acc   = -1.0
best_state = None
no_improve = 0

for epoch in range(EPOCHS):
    model.train()
    for xb, yf, yt, ys, mk in tr_loader:
        xb, yf, yt, ys, mk = xb.to(DEVICE), yf.to(DEVICE), yt.to(DEVICE), ys.to(DEVICE), mk.to(DEVICE)
        lf, lt, ps = model(xb)
        loss = ce_from(lf, yf)
        if mk.any():
            loss = loss + ce_to(lt[mk], yt[mk])
            loss = loss + 0.01 * ((ps[mk] - ys[mk]) ** 2).mean()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    af, at, ms = eval_epoch(va_loader)
    scheduler.step(af)

    if af > best_acc:
        best_acc   = af
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1

    if (epoch + 1) % 10 == 0 or no_improve == 0:
        print(f"Epoch {epoch+1:3d}  from={af:.3f}  to={at:.3f}  mae={ms:.1f}  "
              f"best={best_acc:.3f}  no_improve={no_improve}")

    if no_improve >= PATIENCE_ES:
        print(f"Early stopping at epoch {epoch+1}")
        break

# Restore best weights
model.load_state_dict(best_state)
print(f"\nTraining complete. Best from_val_acc = {best_acc:.3f}")

In [ ]:
out_path = OUTPUT_DIR / "orbit_mlp.pt"
torch.save({"state_dict": model.state_dict(), "input_dim": input_dim}, out_path)
print(f"Saved: {out_path}")
print(f"  input_dim : {input_dim}")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Smoke test: reload and run inference on one batch
ckpt = torch.load(out_path, map_location="cpu", weights_only=False)
m2   = OrbitMLP(ckpt["input_dim"])
m2.load_state_dict(ckpt["state_dict"])
m2.eval()
x_test = torch.randn(2, ckpt["input_dim"])
lf, lt, ps = m2(x_test)
assert lf.shape == (2, 45) and lt.shape == (2, 44) and ps.shape == (2,)
print("Reload smoke test passed.")